# Lab 1 · Exercise A — Gaussian, toy data, fits

**Goals:** build a pdf, generate toy data, fit by maximum likelihood (unbinned and binned), plot.
Fill every cell marked `TODO`, run the notebook top to bottom, and answer the three questions at the end.
Reference: RooFit tutorials `rf101_basics`, `rf201`, `rf202`.

In [ ]:
import ROOT
import numpy as np

MATRICOLA = 123456   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## A1 · Unbinned fit of a Gaussian
Gaussian with mean = 0 and σ = 1. Plot it. Change σ to 3. Generate 10 000 **unbinned** events, fit by ML, plot.

In [ ]:
x     = ROOT.RooRealVar("x", "x", -10, 10)
mean  = ROOT.RooRealVar("mean", "mean", 0, -10, 10)
sigma = ROOT.RooRealVar("sigma", "sigma", 1, 0.1, 10)    # start with sigma = 1

gauss = ROOT.RooGaussian("gauss", "gauss", x, mean, sigma)

frame0 = x.frame(Title="Gaussian pdf: sigma = 1 and sigma = 3")
gauss.plotOn(frame0)
sigma.setVal(3)
gauss.plotOn(frame0, LineColor=ROOT.kRed)
c0 = ROOT.TCanvas("c0", "c0", 800, 500)
frame0.Draw()
c0.Draw()

In [ ]:
# sigma is now 3: generate and fit
data   = gauss.generate(x, 10000)
result = gauss.fitTo(data, Save=True, PrintLevel=-1)

frame1 = x.frame(Title="Unbinned ML fit")
data.plotOn(frame1)
gauss.plotOn(frame1)
c1 = ROOT.TCanvas("c1", "c1", 800, 500)
frame1.Draw()
c1.Draw()
mean.Print(); sigma.Print()

In [ ]:
check("pdf and data exist", gauss is not None and data is not None and result is not None)
check("fit converged", result.status() == 0)
check("sigma compatible with truth (3)", abs(sigma.getVal() - 3) < 4 * sigma.getError())
check("mean compatible with truth (0)", abs(mean.getVal() - 0) < 4 * mean.getError())

## A2 · Binned fit of an exponential
Exponential pdf with rate = −1/τ, τ = 4.5 (mean life). Generate a **binned** dataset of 1000 events,
bin width 0.5. Fit by ML and plot.
*Tip:* express the rate with a `RooFormulaVar("rate", "-1./@0", ROOT.RooArgList(tau))`;
set the number of bins with `t.setBins(...)` **before** `generateBinned`.

In [ ]:
t    = ROOT.RooRealVar("t", "t", 0, 10)
tau  = ROOT.RooRealVar("tau", "mean life", 4.5, 0.5, 10)
rate = ROOT.RooFormulaVar("rate", "-1./@0", ROOT.RooArgList(tau))
expo = ROOT.RooExponential("expo", "expo", t, rate)

t.setBins(20)              # bin width 0.5
data_b = expo.generateBinned(t, 1000)
tau.setVal(2.0)            # start the fit from a wrong value
res_b = expo.fitTo(data_b, Save=True, PrintLevel=-1)

frame2 = t.frame(Title="Binned ML fit")
data_b.plotOn(frame2)
expo.plotOn(frame2)
c2 = ROOT.TCanvas("c2", "c2", 800, 500)
frame2.Draw()
c2.Draw()
tau.Print()

In [ ]:
check("binned dataset built", data_b is not None and res_b is not None)
check("fit converged", res_b.status() == 0)
check("tau compatible with truth (4.5)", abs(tau.getVal() - 4.5) < 4 * tau.getError())

## Questions (answer in this cell, 2–3 lines each)
1. Are the fitted values and errors compatible with the true ones? How did you decide?
2. Repeat A1 with 100 events. What happens to the errors? Is the scaling what you expect?
3. Why is the error on σ different from the error on the mean?

In [ ]:
# Question 2: repeat A1 with only 100 events and compare the errors
err_mean_big, err_sigma_big = mean.getError(), sigma.getError()
sigma.setVal(3); mean.setVal(0)
data_small = gauss.generate(x, 100)
gauss.fitTo(data_small, PrintLevel=-1)
print(f"N=10000: err(mean)={err_mean_big:.4f}  err(sigma)={err_sigma_big:.4f}")
print(f"N=100  : err(mean)={mean.getError():.4f}  err(sigma)={sigma.getError():.4f}")
print("ratio of errors (expect ~ sqrt(10000/100) = 10):", mean.getError() / err_mean_big)

**Solution notes (published after the deadline):**
1. Compare |fit − truth| with the fit error (pull); within ~2σ is normal, beyond ~4σ signals a problem.
2. Errors scale like 1/√N: 100 times fewer events → errors about 10 times larger.
3. For a Gaussian, σ(mean) ≈ σ/√N and σ(σ̂) ≈ σ/√(2N): the error on σ is smaller by a factor √2.